In [94]:
import sqlite3
import re

DB_PATH = "data/flight_database.db"
conn = sqlite3.connect(DB_PATH)

In [95]:
def analyze_sql_validity(pred_path, error_fn):
    preds = [l.strip() for l in open(pred_path)]

    errors = []
    conn = sqlite3.connect(DB_PATH)
    
    for i, pred in enumerate(preds):
        status, error = error_fn(pred, conn)
        
        if status:
            errors.append([i, pred])

    conn.close()

    print(f"Total predictions: {len(preds)}")
    print(f"Total errors: {len(errors)}")

    return errors

In [96]:
# check for syntax errors
def check_sql_syntax(sql, conn):
    try:
        conn.execute("EXPLAIN QUERY PLAN " + sql)
        return False, ""

    except sqlite3.Error as e:
        error = str(e).lower()

        if "syntax error" in error:
            return True, str(e)

        return False, str(e)

In [97]:
from pprint import pprint

syntax_errors = analyze_sql_validity('results/t5_scr_ec_lr3e4_beam_dev.sql', check_sql_syntax)
pprint(syntax_errors[:3])

Total predictions: 466
Total errors: 28
[[2,
  'SELECT DISTINCT flight_1.flight_id FROM flight flight_1, airport_service '
  'airport_service_1, city city_1, airport_service airport_service_2, city '
  'city_2, days days_1, date_day date_day_1 WHERE flight_1.from_airport = '
  'airport_service_1.airport_code AND airport_service_1.city_code = '
  "city_1.city_code AND city_1.city_name = 'BALTIMORE' AND( "
  'flight_1.to_airport = airport_service_2.airport_code AND '
  'airport_service_2.city_code = city_2.city_code AND city_2.city_name = '
  "'BALTIMORE' AND( flight_1.flight_days = days_1.days_code AND "
  'days_1.day_name = date_day_1.day_name AND date_day_1.year = 1991 AND '
  'date_day_1.month_number = 5 AND date_day_1.day_number = 24 AND( '
  'flight_1.arrival_time  flight_1.departure_time AND 1 = 1 ) ) OR( '
  'flight_1.flight_days = days_1.days_code AND days_1.day_name = '
  'date_day_1.day_name AND date'],
 [19,
  'SELECT DISTINCT flight_1.departure_time FROM flight flight_1, '
 

In [98]:
def check_truncation(sql, conn):
    # check for unclosed parentheses
    depth = 0
    in_string = False
    i = 0

    while i < len(sql):
        char = sql[i]

        if char == "'": 
            if in_string and i+1 < len(sql) and sql[i+1] == "'": # close string
                i += 2
                continue
            in_string = not in_string

        elif not in_string:
            if char == '(':
                depth += 1
            elif char == ')':
                depth -= 1

        i += 1

    if in_string:
        return True, "Unclosed string"

    if depth > 0:
        return True, "Unclosed parentheses"

    # check for sql incomplete input error
    try:
        conn.execute("EXPLAIN QUERY PLAN " + sql)

    except sqlite3.Error as e:
        if "incomplete input" in str(e).lower():
            return True, str(e)

    return False, ""

In [99]:
trunc_errors = analyze_sql_validity('results/t5_scr_ec_lr3e4_beam_dev.sql', check_truncation)
pprint(trunc_errors[:3])

Total predictions: 466
Total errors: 69
[[2,
  'SELECT DISTINCT flight_1.flight_id FROM flight flight_1, airport_service '
  'airport_service_1, city city_1, airport_service airport_service_2, city '
  'city_2, days days_1, date_day date_day_1 WHERE flight_1.from_airport = '
  'airport_service_1.airport_code AND airport_service_1.city_code = '
  "city_1.city_code AND city_1.city_name = 'BALTIMORE' AND( "
  'flight_1.to_airport = airport_service_2.airport_code AND '
  'airport_service_2.city_code = city_2.city_code AND city_2.city_name = '
  "'BALTIMORE' AND( flight_1.flight_days = days_1.days_code AND "
  'days_1.day_name = date_day_1.day_name AND date_day_1.year = 1991 AND '
  'date_day_1.month_number = 5 AND date_day_1.day_number = 24 AND( '
  'flight_1.arrival_time  flight_1.departure_time AND 1 = 1 ) ) OR( '
  'flight_1.flight_days = days_1.days_code AND days_1.day_name = '
  'date_day_1.day_name AND date'],
 [15,
  'SELECT DISTINCT flight_1.flight_id FROM flight flight_1, airport_

In [100]:
from collections import Counter

DB_PATH = 'data/flight_database.db'

def execute_sql(sql, conn):
    try:
        result = conn.execute(sql).fetchall()
        return result, None
    except sqlite3.Error as e:
        return None, str(e)

def compare_outputs(gold, pred, conn):
    gold_result, gold_error = execute_sql(gold, conn)
    pred_result, pred_error = execute_sql(pred, conn)

    if gold_error:
        return "GOLD ERROR", gold_error

    if pred_error:
        return "PREDICTION ERROR", pred_error

    if Counter(gold_result) == Counter(pred_result):
        return "MATCH", None

    return "MISMATCH", None

In [101]:
def analyze_output_differences(name, pred_path):

    golds = [l.strip() for l in open("data/dev.sql")]
    preds = [l.strip() for l in open(pred_path)]
    nls = [l.strip() for l in open("data/dev.nl")]

    conn = sqlite3.connect(DB_PATH)

    matches = []
    mismatches = []
    prediction_errors = []
    gold_errors = []

    for i, (nl, gold, pred) in enumerate(zip(nls, golds, preds)):

        status, error = compare_outputs(gold, pred, conn)

        if status == "MATCH":
            matches.append(i)

        elif status == "MISMATCH":
            mismatches.append((i, nl, gold, pred))

        elif status == "PREDICTION ERROR":
            prediction_errors.append((i, gold, pred, error))

        elif status == "GOLD ERROR":
            gold_errors.append((i, gold, error))

    conn.close()

    total = len(golds)

    print(f"\n===== {name} =====")
    print(f"Total queries: {total}")
    print(f"Matching outputs: {len(matches)}/{total}")
    print(f"Different outputs: {len(mismatches)}/{total}")
    print(f"Prediction execution errors: {len(prediction_errors)}/{total}")
    print(f"Gold execution errors: {len(gold_errors)}/{total}")

    return {
        'matches': matches,
        'mismatches': mismatches,
        'prediction_errors': prediction_errors,
        'gold_errors': gold_errors
    }

In [102]:
ft_results = analyze_output_differences('Fine-tuned (lr=0.002)','results/t5_scr_ec_lr3e4_beam_dev.sql')


===== Fine-tuned (lr=0.002) =====
Total queries: 466
Matching outputs: 223/466
Different outputs: 147/466
Prediction execution errors: 96/466
Gold execution errors: 0/466


In [ ]:
# individually look through mismatches and classify error
ft_results["mismatches"]

[(5,
  "i'm starting from denver",
  "SELECT DISTINCT flight_1.flight_id FROM flight flight_1 , airport_service airport_service_1 , city city_1 WHERE flight_1.from_airport = airport_service_1.airport_code AND airport_service_1.city_code = city_1.city_code AND city_1.city_name = 'DENVER'",
  "SELECT DISTINCT flight_1.flight_id FROM flight flight_1, airport_service airport_service_1, city city_1, airport_service airport_service_2, city city_2 WHERE flight_1.from_airport = airport_service_1.airport_code AND airport_service_1.city_code = city_1.city_code AND city_1.city_name = 'DENVER' AND flight_1.to_airport = airport_service_2.airport_code AND airport_service_2.city_code = city_2.city_code AND city_2.city_name = 'DENVER'"),
 (8,
  'show me the flights from baltimore to atlanta',
  "SELECT DISTINCT flight_1.flight_id FROM flight flight_1 , airport_service airport_service_1 , city city_1 , airport_service airport_service_2 , city city_2 WHERE flight_1.from_airport = airport_service_1.airpo

In [ ]:
for (i, nl, gold, pred) in ft_results["mismatches"]:
    if i in [69, 100, 106, 150, 158, 171, 173, 178, 229, 286, 290, 313, 394, 411, 426]: # ft
        print((nl, gold, pred))

('please list nonstop flights from las vegas to new york on america west', "SELECT DISTINCT flight_1.flight_id FROM flight flight_1 , airport_service airport_service_1 , city city_1 , airport_service airport_service_2 , city city_2 WHERE flight_1.airline_code = 'HP' AND( flight_1.stops = 0 AND( flight_1.from_airport = airport_service_1.airport_code AND airport_service_1.city_code = city_1.city_code AND city_1.city_name = 'LAS VEGAS' AND flight_1.to_airport = airport_service_2.airport_code AND airport_service_2.city_code = city_2.city_code AND city_2.city_name = 'NEW YORK' ) )", "SELECT DISTINCT flight_1.flight_id FROM flight flight_1, airport_service airport_service_1, city city_1, airport_service airport_service_2, city city_2 WHERE flight_1.stops = 0 AND( flight_1.from_airport = airport_service_1.airport_code AND airport_service_1.city_code = city_1.city_code AND city_1.city_name = 'LAS VEGAS' AND flight_1.to_airport = airport_service_2.airport_code AND airport_service_2.city_code = 

In [107]:
for (i, nl, gold, pred) in ft_results["mismatches"]:
    if i in [20, 52, 69, 79, 90, 132, 133, 140, 150, 156, 158, 174, 181, 186, 206, 233, 241, 313, 379, 411, 450]: # ec
        print((nl, gold, pred))

('list all flights from boston to san francisco with the maximum number of stops', "SELECT DISTINCT flight_1.flight_id FROM flight flight_1 , airport_service airport_service_1 , city city_1 , airport_service airport_service_2 , city city_2 WHERE flight_1.from_airport = airport_service_1.airport_code AND airport_service_1.city_code = city_1.city_code AND city_1.city_name = 'BOSTON' AND( flight_1.to_airport = airport_service_2.airport_code AND airport_service_2.city_code = city_2.city_code AND city_2.city_name = 'SAN FRANCISCO' AND flight_1.stops =( SELECT MAX( stops ) FROM flight flight_1 , airport_service airport_service_1 , city city_1 , airport_service airport_service_2 , city city_2 WHERE flight_1.from_airport = airport_service_1.airport_code AND airport_service_1.city_code = city_1.city_code AND city_1.city_name = 'BOSTON' AND flight_1.to_airport = airport_service_2.airport_code AND airport_service_2.city_code = city_2.city_code AND city_2.city_name = 'SAN FRANCISCO' ) )", "SELECT 